处理openai_clauses_pp_txt的结果，
根据1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\privacy_policy_md\pp_download_results.csv知道同一个apkname如果有相同的link就跳过md下载，记录为duplicated=1.

那么在1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch里面，每一个文件夹名字是一个apkname，在pp_download_results.csv能找到对应的两行数据，一行对应一个json，json文件名是apkname和version，如果有duplicated=1，就把该json文件copy一份然后命名为apkname和该duplicated=1所在行的version

如果是同一个apkname duplicated，那么就复制一份一模一样的结果，版本号不一样而已

com.breakingnewsbrief.app empty
com.bydeluxe.d3.android.program.starz --不知道为啥一个版本能对应上，17版本对应不上regions-因为这个版本值为空。
com.EternalStudio.SurvivorZ 均不重复，3个版本
com.halfbrick.fruitninjax 不知道为啥没有子矩阵，3个版本
com.hecorat.screenrecorder.free -- 3个版本，29/35/37
com.herocraft.game.free.stww2_sandbox -- 3个版本
com.hiroba.helix -- 3个版本

"com.EternalStudio.SurvivorZ", "com.halfbrick.fruitninjax", "com.hecorat.screenrecorder.free", "com.herocraft.game.free.stww2_sandbox", "com.hiroba.helix"

In [7]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA6_sisth_74_batch" # , AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch

In [8]:
from pathlib import Path
import pandas as pd
import shutil
import re

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA4_forth_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA5_fifth_100_batch\privacy_policy_md
# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA6_sisth_74_batch\privacy_policy_md
csv_path = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\privacy_policy_md\pp_download_results.csv")

# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch
json_root = Path(rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\{f_position}\{s_position}")

df = pd.read_csv(csv_path)

df.head()

,apk_name,version,privacy_policy_url,apkitself_url,http_status,success,saved_path,content_length,error,duplicated,status
0,com.lutech.theme,129,https://web.archive.org/web/20240813184936/htt...,https://web.archive.org/web/20240813184936/htt...,498.0,False,NaN,0.0,http_498,0,NaN
1,com.lutech.theme,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,empty_url
2,com.lydia,55576,https://sumeria.eu/en/essentials/terms-and-con...,https://sumeria.eu/en/essentials/terms-and-con...,200.0,True,1122apk\1122apk_privacy_policy_url\pp_md_2024_...,78133.0,NaN,0,NaN
3,com.lydia,55589,https://sumeria.eu/en/essentials/terms-and-con...,NaN,NaN,NaN,NaN,NaN,NaN,1,duplicate_url_same_apk
4,com.mabuk.money.duit,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,empty_url


In [9]:
# 如果 duplicated 是字符串，统一转成数字
df["duplicated"] = df["duplicated"].astype(str).str.strip().astype(int)

# 根据你的 csv 列名调整这里
APK_COL = "apk_name"
VERSION_COL = "version"
LINK_COL = "privacy_policy_url"  


def safe_name(s: str) -> str:
    """
    用于兼容文件名里的特殊字符。
    如果你原始 json 文件名没有做这个处理，可以删掉这个函数相关逻辑。
    """
    s = str(s)
    return re.sub(r'[<>:"/\\|?*]', "_", s)


def find_json(apk_dir: Path, apkname: str, version: str):
    """
    在 apkname 文件夹里寻找对应 version 的 json。
    兼容几种可能命名：
    1. {apkname}_{version}.json
    2. {safe_apkname}_{safe_version}.json
    3. 文件名中同时包含 apkname 和 version
    """
    candidates = [
        apk_dir / f"{apkname}_{version}.json",
        apk_dir / f"{safe_name(apkname)}_{safe_name(version)}.json",
    ]

    for p in candidates:
        if p.exists():
            return p

    # fallback：在目录里搜索包含 version 的 json
    version_s = str(version)
    all_jsons = list(apk_dir.glob("*.json"))
    matched = [p for p in all_jsons if version_s in p.name]

    if len(matched) == 1:
        return matched[0]

    # 再 fallback：如果该 apk 文件夹里只有一个 json，也可以作为源文件
    if len(all_jsons) == 1:
        return all_jsons[0]

    return None


def target_json_path(apk_dir: Path, apkname: str, version: str):
    """
    目标文件名。这里按 apkname_version.json 命名。
    如果你原来的 json 命名规则不同，改这里即可。
    """
    return apk_dir / f"{apkname}_{version}.json"


copied = []
skipped = []
errors = []

for apkname, g in df.groupby(APK_COL):
    apk_dir = json_root / str(apkname)

    if not apk_dir.exists():
        errors.append((apkname, "apk folder not found", str(apk_dir)))
        continue

    dup_rows = g[g["duplicated"] == 1]
    non_dup_rows = g[g["duplicated"] != 1]

    if dup_rows.empty:
        continue

    if non_dup_rows.empty:
        errors.append((apkname, "duplicated rows exist but no non-duplicated source row", ""))
        continue

    for _, dup_row in dup_rows.iterrows():
        dup_version = str(dup_row[VERSION_COL])
        print(f"Processing {apkname} version {dup_version} (duplicated)")

        # 优先找同 link 的非 duplicated 行
        source_rows = non_dup_rows
        if LINK_COL in df.columns:
            same_link_rows = non_dup_rows[
                non_dup_rows[LINK_COL].astype(str) == str(dup_row[LINK_COL])
            ]
            if not same_link_rows.empty:
                source_rows = same_link_rows

        # 通常只有一行，如果有多行，取第一行
        src_row = source_rows.iloc[0]
        src_version = str(src_row[VERSION_COL])

        src_json = find_json(apk_dir, str(apkname), src_version)
        if src_json is None:
            errors.append((apkname, f"source json not found for version {src_version}", ""))
            continue

        dst_json = target_json_path(apk_dir, str(apkname), dup_version)
        print(f"Copying from {src_json} to {dst_json}")

        if dst_json.exists():
            skipped.append((apkname, dup_version, "target already exists", str(dst_json)))
            continue

        shutil.copy2(src_json, dst_json)
        copied.append((apkname, src_version, dup_version, str(src_json), str(dst_json)))


print(f"Copied: {len(copied)}")
print(f"Skipped: {len(skipped)}")
print(f"Errors: {len(errors)}")

if copied:
    print("\nCopied examples:")
    for item in copied[:10]:
        print(item)

if skipped:
    print("\nSkipped examples:")
    for item in skipped[:10]:
        print(item)

if errors:
    print("\nErrors:")
    for item in errors[:20]:
        print(item)

Processing com.lydia version 55589 (duplicated)
Copying from 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.lydia\com.lydia_55576.json to 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.lydia\com.lydia_55589.json
Processing com.magdalm.freewifipassword version 1403 (duplicated)
Copying from 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.magdalm.freewifipassword\com.magdalm.freewifipassword_1401.json to 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.magdalm.freewifipassword\com.magdalm.freewifipassword_1403.json
Processing com.map.photostamp version 106 (duplicated)
Copying from 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA6_sisth_74_batch\com.map.photostamp\com.map.photostamp_100.json to 1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_

com.breakingnewsbrief.app
com.bydeluxe.d3.android.program.starz --
com.EternalStudio.SurvivorZ
com.halfbrick.fruitninjax
com.hecorat.screenrecorder.free --
com.herocraft.game.free.stww2_sandbox --
com.hiroba.helix
